# Choosing a model, an endpoint, and an API on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

A live capability survey across every family on `bedrock-mantle`. Rather than
trusting a table someone wrote months ago, this notebook **probes the endpoint**
and builds the table from what the API actually does today.

Use it to answer: *which model, which API, which path, which parameters?*

## What this notebook produces
- The full model inventory and its Region footprint
- A per-family API matrix (Responses / Chat Completions / Messages)
- A per-model parameter matrix (`temperature`, `top_p`, `service_tier`, …)
- A reusable `capabilities.py` you can drop into your own project

## Self-contained, but see also
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- Family deep-dives: `../01-openai-gpt/` … `../12-writer-palmyra/`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import concurrent.futures as cf
import json
import re
import sys
from collections import defaultdict

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, ok, post, runtime_post

REGION = "us-east-1"  # the widest inventory
REGIONS = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
print("survey region:", REGION)

survey region: us-east-1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-7",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
    "deepseek.v3.2",
    "google.gemma-4-31b",
    "minimax.minimax-m2.5",
    "mistral.mistral-large-3-675b-instruct",
    "moonshotai.kimi-k2.5",
    "nvidia.nemotron-super-3-120b",
    "openai.gpt-5.5",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "openai.gpt-oss-safeguard-20b",
    "qwen.qwen3-32b",
    "writer.palmyra-vision-7b",
    "xai.grok-4.3",
    "zai.glm-5",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime


anthropic.claude-opus-4-7              us.anthropic.claude-opus-4-7             mantle, runtime
anthropic.claude-opus-4-8              us.anthropic.claude-opus-4-8             mantle, runtime
anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime


anthropic.claude-sonnet-5              us.anthropic.claude-sonnet-5             mantle, runtime


deepseek.v3.2                          deepseek.v3.2                            mantle, runtime


...

qwen.qwen3-32b                         qwen.qwen3-32b-v1:0                      mantle, runtime
writer.palmyra-vision-7b               writer.palmyra-vision-7b                 mantle, runtime


xai.grok-4.3                           -- not on runtime --                     mantle


zai.glm-5                              zai.glm-5                                mantle, runtime


  [bedrock helper] could not list bedrock-mantle models in us-east-1 (RuntimeError). Every 'mantle' answer below is False because the catalogue is unavailable, NOT because the model is absent.



=> 16/19 of these are on bedrock-runtime; 8 under a different id.
   bedrock-mantle only: ['google.gemma-4-31b', 'openai.gpt-5.5', 'xai.grok-4.3']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. Inventory and Region footprint

In [3]:
inventory = {}
for region in REGIONS:
    try:
        inventory[region] = sorted(list_models(region))
    except (RuntimeError, OSError) as exc:
        inventory[region] = []
        print(f"{region}: {type(exc).__name__}")

for region in REGIONS:
    print(f"{region:14} {len(inventory[region]):3} models")

models = inventory[REGION]
families = defaultdict(list)
for mid in models:
    families[mid.split(".")[0]].append(mid)

print(f"\n{len(models)} models across {len(families)} families in {REGION}")

us-east-1       55 models
us-east-2       50 models
us-west-2       49 models
eu-central-1    33 models

55 models across 12 families in us-east-1


In [4]:
print(f"{'family':14} {'n':>3}  models")
print("-" * 100)
for family in sorted(families):
    ids = families[family]
    print(
        f"{family:14} {len(ids):>3}  {', '.join(i.split('.', 1)[1] for i in ids)[:76]}"
    )

family           n  models
----------------------------------------------------------------------------------------------------
anthropic        6  claude-fable-5, claude-haiku-4-5, claude-opus-4-7, claude-opus-4-8, claude-o
deepseek         2  v3.1, v3.2
google           6  gemma-3-12b-it, gemma-3-27b-it, gemma-3-4b-it, gemma-4-26b-a4b, gemma-4-31b,
minimax          3  minimax-m2, minimax-m2.1, minimax-m2.5
mistral          8  devstral-2-123b, magistral-small-2509, ministral-3-14b-instruct, ministral-3
moonshotai       2  kimi-k2-thinking, kimi-k2.5
nvidia           4  nemotron-nano-12b-v2, nemotron-nano-3-30b, nemotron-nano-9b-v2, nemotron-sup
openai          11  gpt-5.4, gpt-5.4-2026-03-05, gpt-5.5, gpt-5.5-2026-04-23, gpt-5.6-luna, gpt-
qwen             7  qwen3-235b-a22b-2507, qwen3-32b, qwen3-coder-30b-a3b-instruct, qwen3-coder-4
writer           1  palmyra-vision-7b
xai              1  grok-4.3
zai              4  glm-4.6, glm-4.7, glm-4.7-flash, glm-5


In [5]:
# Which models are missing where? This is the deployment-planning view.
everywhere = set(inventory[REGIONS[0]])
for region in REGIONS[1:]:
    everywhere &= set(inventory[region])
print(f"available in ALL four Regions: {len(everywhere)}")

only_one = [m for m in models if sum(1 for r in REGIONS if m in inventory[r]) == 1]
print(f"available in only ONE Region : {len(only_one)}")
for mid in sorted(only_one):
    where = [r for r in REGIONS if mid in inventory[r]]
    print(f"   {mid:44} {where[0]}")

available in ALL four Regions: 33
available in only ONE Region : 5
   anthropic.claude-fable-5                     us-east-1
   anthropic.claude-opus-4-7                    us-east-1
   anthropic.claude-opus-4-8                    us-east-1
   anthropic.claude-opus-5                      us-east-1
   anthropic.claude-sonnet-5                    us-east-1


## 2. Path resolution — per endpoint, not just per model

Prefixes depend on the model family **and** on the endpoint, and the split falls in
a different place on each:

| | `bedrock-mantle` | `bedrock-runtime` |
|---|---|---|
| `/openai/v1` | `google.gemma-4*`, `xai.*`, and the hosted `openai.gpt-*` models | **every** OpenAI-compatible model |
| `/v1` | everything else non-Anthropic | **does not exist** |
| `/anthropic/v1` | `anthropic.*` | `anthropic.*` |

So `openai.gpt-oss-120b` is `/v1` on mantle and its runtime twin
`openai.gpt-oss-120b-1:0` is `/openai/v1`. A resolver that takes only a model ID can
be right about one endpoint at a time — this collection shipped exactly that until
`bedrock-runtime` grew the OpenAI-compatible paths in August 2026.

Copy the version below, with the `endpoint` argument.

In [6]:
import re


PROFILE_RE = re.compile(r"^(us|eu|apac|global|in)\.")


def api_prefix(model_id: str, endpoint: str = "mantle") -> str:
    """Which URL prefix serves this model's inference APIs, on this endpoint?"""
    # Strip a geo/global inference-profile prefix first: "us.anthropic.claude-opus-5"
    # does not start with "anthropic.", and a naive check routes it to /openai/v1.
    bare = PROFILE_RE.sub("", model_id)
    if bare.startswith("anthropic."):
        return "/anthropic/v1"
    if endpoint == "runtime":
        return "/openai/v1"
    if bare.startswith(("google.gemma-4", "openai.gpt-5", "xai.")):
        return "/openai/v1"
    return "/v1"


by_prefix = defaultdict(list)
for mid in models:
    by_prefix[api_prefix(mid)].append(mid)

print("on bedrock-mantle:")
for prefix in sorted(by_prefix):
    ids = by_prefix[prefix]
    fams = sorted({i.split(".")[0] for i in ids})
    print(f"  {prefix:16} {len(ids):3} models | families: {', '.join(fams)}")

# The same models, addressed on bedrock-runtime. Note that the /v1 bucket empties.
runtime_ids = [r for m in models if (r := runtime_id_for(m, REGION)) is not None]
runtime_by_prefix = defaultdict(list)
for rid in runtime_ids:
    runtime_by_prefix[api_prefix(rid, "runtime")].append(rid)

print(f"\non bedrock-runtime ({len(runtime_ids)} of {len(models)} are there):")
for prefix in sorted(runtime_by_prefix):
    ids = runtime_by_prefix[prefix]
    fams = sorted({PROFILE_RE.sub("", i).split(".")[0] for i in ids})
    print(f"  {prefix:16} {len(ids):3} models | families: {', '.join(fams)}")

moved = [
    m for m in models
    if (r := runtime_id_for(m, REGION)) is not None
    and api_prefix(m) != api_prefix(r, "runtime")
]
print(f"\n=> {len(moved)} model(s) change PATH between endpoints, e.g. "
      f"{moved[:3]}")
print(f"=> bedrock-runtime serves no /v1 inference path at all: "
      f"{'/v1' not in runtime_by_prefix}")

on bedrock-mantle:
  /anthropic/v1      6 models | families: anthropic
  /openai/v1        11 models | families: google, openai, xai
  /v1               38 models | families: deepseek, google, minimax, mistral, moonshotai, nvidia, openai, qwen, writer, zai

on bedrock-runtime (43 of 55 are there):
  /anthropic/v1      6 models | families: anthropic
  /openai/v1        37 models | families: deepseek, google, minimax, mistral, moonshot, moonshotai, nvidia, openai, qwen, writer, zai

=> 34 model(s) change PATH between endpoints, e.g. ['deepseek.v3.2', 'google.gemma-3-12b-it', 'google.gemma-3-27b-it']
=> bedrock-runtime serves no /v1 inference path at all: True


Note the split *inside* the OpenAI family: the provider name alone is not enough, and
neither is the version. Keying this on `gpt-5.*` is what broke when GPT-6 Astra
arrived, because a generation bump moved the model to a prefix that does not serve it.
The line is the discriminator, not the number: `01-openai-gpt/01` §2 measures both
prefixes for each OpenAI model mantle lists and prints the rows where the resolver and
the service disagree.

In [7]:
for mid in sorted(families["openai"]):
    print(f"  {mid:34} -> {api_prefix(mid)}")

  openai.gpt-5.4                     -> /openai/v1
  openai.gpt-5.4-2026-03-05          -> /openai/v1
  openai.gpt-5.5                     -> /openai/v1
  openai.gpt-5.5-2026-04-23          -> /openai/v1
  openai.gpt-5.6-luna                -> /openai/v1
  openai.gpt-5.6-sol                 -> /openai/v1
  openai.gpt-5.6-terra               -> /openai/v1
  openai.gpt-oss-120b                -> /v1
  openai.gpt-oss-20b                 -> /v1
  openai.gpt-oss-safeguard-120b      -> /v1
  openai.gpt-oss-safeguard-20b       -> /v1


## 3. Probe the API surface per family

One representative per family, three APIs each. Run in parallel to keep it quick.

In [8]:
REPRESENTATIVES = [
    "google.gemma-4-31b",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "anthropic.claude-haiku-4-5",
    "xai.grok-4.3",
    "qwen.qwen3-32b",
    "deepseek.v3.2",
    "zai.glm-5",
    "minimax.minimax-m2.5",
    "moonshotai.kimi-k2.5",
    "mistral.mistral-large-3-675b-instruct",
    "nvidia.nemotron-super-3-120b",
    "writer.palmyra-vision-7b",
]
REPRESENTATIVES = [m for m in REPRESENTATIVES if m in models]
AV = {"anthropic-version": "2023-06-01"}


def probe_apis(model_id: str) -> dict:
    """Status code per API. Single attempt + short timeout: a wrong path can stall."""
    prefix = api_prefix(model_id)
    out = {"model": model_id, "prefix": prefix}

    if prefix == "/anthropic/v1":
        code, _ = post(
            f"{prefix}/messages",
            {
                "model": model_id,
                "max_tokens": 16,
                "messages": [{"role": "user", "content": "Hi"}],
            },
            region=REGION,
            headers=AV,
            attempts=1,
            timeout=45,
        )
        out["messages"] = code
        out["responses"] = out["chat"] = "-"
        out["chat_field"] = "-"
        return out

    # Try BOTH budget field names before concluding an API is missing. gpt-5.6
    # serves Chat Completions but refuses `max_tokens`, and reading that 400 as
    # "no Chat Completions" is how a false claim reached three other notebooks.
    code, _ = post(
        f"{prefix}/responses",
        {"model": model_id, "input": "Hi", "max_output_tokens": 16},
        region=REGION,
        attempts=1,
        timeout=45,
    )
    out["responses"] = code
    for field in ("max_tokens", "max_completion_tokens"):
        code, _ = post(
            f"{prefix}/chat/completions",
            {
                "model": model_id,
                "messages": [{"role": "user", "content": "Hi"}],
                field: 16,
            },
            region=REGION,
            attempts=1,
            timeout=45,
        )
        if code == 200:
            out["chat_field"] = field
            break
    out["chat"] = code
    out.setdefault("chat_field", "-")
    out["messages"] = "-"
    return out


with cf.ThreadPoolExecutor(max_workers=5) as pool:
    api_results = list(pool.map(probe_apis, REPRESENTATIVES))

print(f"{'model':40} {'prefix':16} {'Resp':>6} {'Chat':>6} {'Msg':>6} {'CC budget':>22}")
print("-" * 104)
for row in api_results:
    print(
        f"{row['model']:40} {row['prefix']:16} {str(row['responses']):>6} "
        f"{str(row['chat']):>6} {str(row['messages']):>6} "
        f"{str(row.get('chat_field', '-')):>22}"
    )

model                                    prefix             Resp   Chat    Msg              CC budget
--------------------------------------------------------------------------------------------------------
google.gemma-4-31b                       /openai/v1          200    200      -             max_tokens
openai.gpt-5.6-sol                       /openai/v1          200    200      -  max_completion_tokens
openai.gpt-oss-120b                      /v1                 200    200      -             max_tokens
anthropic.claude-haiku-4-5               /anthropic/v1         -      -    200                      -
xai.grok-4.3                             /openai/v1          200    200      -             max_tokens
qwen.qwen3-32b                           /v1                 400    200      -             max_tokens
deepseek.v3.2                            /v1                 400    200      -             max_tokens
zai.glm-5                                /v1                 400    200      - 

`200` means available; `400` means "this model does not serve that API"; `-1`
means the request **stalled** rather than erroring — which is why every probe
above sets a timeout.

In [9]:
summary = {"responses": [], "chat": [], "messages": []}
for row in api_results:
    for key in summary:
        if row[key] == 200:
            summary[key].append(row["model"].split(".")[0])
print("Responses API       :", sorted(set(summary["responses"])))
print("Chat Completions    :", sorted(set(summary["chat"])))
print("Anthropic Messages  :", sorted(set(summary["messages"])))

Responses API       : ['google', 'openai', 'xai']
Chat Completions    : ['deepseek', 'google', 'minimax', 'mistral', 'moonshotai', 'nvidia', 'openai', 'qwen', 'writer', 'xai', 'zai']
Anthropic Messages  : ['anthropic']


## 4. Parameter compatibility

Sampling parameters are the most common cross-family break. There is **no single
config that works everywhere** — this table proves it.

In [10]:
# Four cases, not two. The paragraph after this cell drew three conclusions the old
# two-case probe could not support: that GPT-5.6 accepts temperature only at its
# DEFAULT (never sent 1.0), that haiku-4-5 accepts either but not both (never sent
# them together), and that newer Claude models refuse both as deprecated (no opus-5
# or sonnet-5 row existed). Each of those now has a column.
CASES = (
    ("temp 0.5", {"temperature": 0.5}),
    ("temp 1.0", {"temperature": 1.0}),
    ("top_p", {"top_p": 0.9}),
    ("both", {"temperature": 0.5, "top_p": 0.9}),
)


def _verdict(code: int) -> str:
    """A refusal, a transient failure, or acceptance -- never conflate them.

    `attempts=1` plus `"ok" if code == 200` read a stall (-1) or a transient 5xx as a
    refusal, and the conclusions below then published a capability claim from a
    network blip: one stall on grok's `both` probe printed
    `accept either but NOT both: ['claude-haiku-4-5', 'xai.grok-4.3']`. The sibling
    cell in section 5 was fixed for exactly this ("attempts=1 wrote '500' into this
    table ... as though the feature were missing") and this one was not. Only a 4xx
    is the service saying no.
    """
    if code == 200:
        return "ok"
    if 400 <= code < 500:
        return str(code)
    return "?"          # -1 stall or 5xx: inconclusive, not a refusal
# REPRESENTATIVES drives several other tables, so widen only this probe. The newer
# Claude models are the subject of one of the conclusions and were absent.
# gpt-5.5 and gpt-5.4 are here because `TEMPERATURE_DEFAULT_ON_RESPONSES` in the
# resolver below is keyed on gpt-5.5 alone, and NOTHING in this collection probed it:
# not this cell, not section 6's verification, not 02's survey. The one rule with a
# per-(model, API) key was the one rule with no measurement.
SAMPLING_MODELS = REPRESENTATIVES + [
    m for m in ("anthropic.claude-opus-5", "anthropic.claude-sonnet-5",
                "openai.gpt-5.5", "openai.gpt-5.4")
    if m in models and m not in REPRESENTATIVES
]


def probe_params(model_id: str) -> dict:
    prefix = api_prefix(model_id)
    out = {"model": model_id}

    if prefix == "/anthropic/v1":
        base = {
            "model": model_id,
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Hi"}],
        }
        for label, extra in CASES:
            code, data = post(
                f"{prefix}/messages",
                {**base, **extra},
                region=REGION,
                headers=AV,
                attempts=2,
                timeout=45,
            )
            out[label] = _verdict(code)
            if code >= 400 and code < 500:
                out.setdefault("why", err(data, limit=120))
        out["service_tier"] = "-"
        return out

    # Use whichever inference API this model actually serves.
    code, _ = post(
        f"{prefix}/responses",
        {"model": model_id, "input": "Hi", "max_output_tokens": 16},
        region=REGION,
        attempts=1,
        timeout=45,
    )
    if code == 200:
        path, base = f"{prefix}/responses", {
            "model": model_id,
            "input": "Hi",
            "max_output_tokens": 16,
        }
    else:
        # Try BOTH budget field names before settling, exactly as section 6 does.
        # Hardcoding max_tokens here meant that for any model requiring
        # max_completion_tokens every probe below returned 400 and the table reported
        # the model as refusing every sampling parameter -- blaming the parameter for
        # a budget-field error, which is the trap this notebook is about.
        path = f"{prefix}/chat/completions"
        base = None
        for field in ("max_tokens", "max_completion_tokens"):
            candidate = {
                "model": model_id,
                "messages": [{"role": "user", "content": "Hi"}],
                field: 16,
            }
            probe_code, _ = post(
                path, candidate, region=REGION, attempts=1, timeout=45
            )
            if probe_code == 200:
                base = candidate
                break
        if base is None:
            # Neither field works: report that rather than mislabelling three
            # parameters as refused.
            out["temperature"] = out["top_p"] = out["service_tier"] = "no base call"
            return out

    for label, extra in (*CASES, ("service_tier", {"service_tier": "flex"})):
        code, data = post(path, {**base, **extra}, region=REGION, attempts=2,
                          timeout=45)
        out[label] = _verdict(code)
        if 400 <= code < 500:
            out.setdefault("why", err(data, limit=120))
    return out


with cf.ThreadPoolExecutor(max_workers=5) as pool:
    param_results = list(pool.map(probe_params, SAMPLING_MODELS))

labels = [label for label, _ in CASES]
header = "".join(f"{label:>10}" for label in labels)
print(f"{'model':40}{header}{'flex tier':>11}")
print("-" * (51 + 10 * len(labels)))
for row in param_results:
    cells = "".join(f"{row.get(label, '-'):>10}" for label in labels)
    print(f"{row['model']:40}{cells}{row.get('service_tier', '-'):>11}")

# The conclusions, computed from the rows above rather than written underneath them.
def _of(model_prefix):
    return [r for r in param_results if r["model"].startswith(model_prefix)]

print("\nderived from the table:")
gpt56 = _of("openai.gpt-5.6")
if gpt56:
    default_only = all(r.get("temp 0.5") != "ok" and r.get("temp 1.0") == "ok"
                       for r in gpt56)
    print(f"  gpt-5.6: temperature at the default 1.0 only -> {default_only}; "
          f"top_p accepted -> {all(r.get('top_p') == 'ok' for r in gpt56)}")
# `not in ("ok", ...)` would count "?" -- an inconclusive stall -- as a refusal, which
# is the bug _verdict() exists to prevent. Require an explicit 4xx.
either_not_both = [r["model"] for r in param_results
                   if r.get("temp 0.5") == "ok" and r.get("top_p") == "ok"
                   and str(r.get("both", "")).isdigit()]
inconclusive = [(r["model"], k) for r in param_results for k in labels
                if r.get(k) == "?"]
print(f"  accept either but NOT both in one request: {either_not_both or 'none'}")
deprecated = [r["model"] for r in param_results
              if "deprecated" in (r.get("why") or "")]
print(f"  refuse sampling as *deprecated*:           {deprecated or 'none'}")
permissive = [r["model"] for r in param_results
              if all(r.get(k) == "ok" for k in labels)]
print(f"  permissive on all four cases:              {len(permissive)} model(s)")
for row in param_results:
    if row.get("why"):
        print(f"    {row['model']:40} {row['why'][:70]}")
if inconclusive:
    print(f"  !! inconclusive (stall or 5xx, NOT a refusal): {inconclusive}")
    print("     Re-run before reading any conclusion above that involves them.")

# The table above probes ONE API per model -- whichever the model serves -- so it
# cannot see a restriction that differs BETWEEN the two APIs a model serves. That is
# exactly the shape of the resolver's `TEMPERATURE_DEFAULT_ON_RESPONSES` rule, so
# probe it head-on: same model, same parameter, both APIs.
print("\nper-API check, for models that serve both Responses and Chat Completions:")
print(f"  {'model':22} {'parameter':16} {'Responses':>10} {'Chat':>10}  per-API?")
BOTH_API = [m for m in ("openai.gpt-5.4", "openai.gpt-5.5", "openai.gpt-5.6-sol")
            if m in models]
per_api_rows = []
for model_id in BOTH_API:
    for pname, extra in (("temperature=0.5", {"temperature": 0.5}),
                         ("top_p=0.9", {"top_p": 0.9})):
        r_code, _ = post(f"/openai/v1/responses",
                         {"model": model_id, "input": "Reply OK",
                          "max_output_tokens": 2000, **extra},
                         region=REGION, attempts=2, timeout=60)
        c_code, _ = post(f"/openai/v1/chat/completions",
                         {"model": model_id,
                          "messages": [{"role": "user", "content": "Reply OK"}],
                          "max_completion_tokens": 2000, **extra},
                         region=REGION, attempts=2, timeout=60)
        differs = (r_code == 200) != (c_code == 200)
        per_api_rows.append((model_id, pname, r_code, c_code, differs))
        print(f"  {model_id:22} {pname:16} {_verdict(r_code):>10} "
              f"{_verdict(c_code):>10}  {'YES' if differs else 'no'}")

per_api = sorted({m for m, _, _, _, d in per_api_rows if d})
print(f"\n=> restriction depends on the API for: {per_api or 'no model in this set'}")
print("   That is why the resolver takes an `api` argument as well as an `endpoint`:")
print("   a rule keyed on the model alone must be wrong for one of the two routes.")

model                                     temp 0.5  temp 1.0     top_p      both  flex tier
-------------------------------------------------------------------------------------------
google.gemma-4-31b                              ok        ok        ok        ok         ok
openai.gpt-5.6-sol                             400        ok       400       400        400
openai.gpt-oss-120b                             ok        ok        ok        ok         ok
anthropic.claude-haiku-4-5                      ok        ok        ok       400          -
xai.grok-4.3                                    ok        ok        ok        ok         ok
qwen.qwen3-32b                                  ok        ok        ok        ok         ok
deepseek.v3.2                                   ok        ok        ok        ok         ok
zai.glm-5                                       ok        ok        ok        ok         ok
minimax.minimax-m2.5                            ok        ok        ok        ok

  openai.gpt-5.4         temperature=0.5          ok         ok  no


  openai.gpt-5.4         top_p=0.9                ok         ok  no


  openai.gpt-5.5         temperature=0.5         400         ok  YES


  openai.gpt-5.5         top_p=0.9               400         ok  YES
  openai.gpt-5.6-sol     temperature=0.5         400        400  no


  openai.gpt-5.6-sol     top_p=0.9               400        400  no

=> restriction depends on the API for: ['openai.gpt-5.5']
   That is why the resolver takes an `api` argument as well as an `endpoint`:
   a rule keyed on the model alone must be wrong for one of the two routes.


Read the table, not this paragraph — the paragraph is the part that goes stale.
When this notebook was first written it claimed Gemma 4 took `temperature` and
refused `top_p`, directly beneath a table showing `temperature` refused. Both the
table and the prose have since been wrong in different directions.

The cell above now prints its own conclusions under **derived from the table**, so
read those lines rather than this paragraph. They exist because the three claims that
used to sit here were not established by the probe above them: it sent `temperature=0.5`
and `top_p=0.9` separately and never `temperature=1.0`, never both together, and never
asked opus-5 or sonnet-5 at all — while the paragraph asserted a default-only rule, an
either-but-not-both rule, and a deprecation for models with no row.

What to take away, none of it model-specific:

- **There is no universal sampling config.** Four different shapes of refusal show up
  in one table: refused outright, refused only at a non-default value, refused only in
  combination, and accepted.
- **`gpt-5.x` and Claude reject `flex`/`priority`.** For Claude, `service_tier` is
  refused at every value except `default` on Messages, so `-` in that column
  means "omitted" rather than "rejected" — §6's resolver check prints what
  it decided and why.
- **A `-1` means the request stalled**, not that anything was rejected. That is why
  every probe here sets a timeout.
- **The refusal message names the parameter**, and the `why` column prints it. That
  is what makes the error-driven approach in `02-migrating-from-openai` §7 work, and
  why it survives table drift that a static rule does not.

Resolve sampling and tier per model, and re-probe: rows here have changed more than
once during this collection's life.

## 5. Structured-output support

In [11]:
SCHEMA = {
    "type": "object",
    "properties": {"answer": {"type": "string"}},
    "required": ["answer"],
    "additionalProperties": False,
}


ASK = [{"role": "user", "content": "Answer 'hi'."}]
# attempts=2: a transient 5xx is not a capability answer, and attempts=1 wrote
# "500" into this table for nemotron as though the feature were missing.
PROBE = {"region": REGION, "attempts": 2, "timeout": 90}


def _claude_forced_tool(model_id: str, prefix: str, samples: int = 3) -> str:
    """Claude: output_config.format is rejected on mantle; forced tools work.

    Reported as a rate, like the other two columns. Claude puts the object in a
    `tool_use` block rather than in `tool_calls`, so the conformance check differs,
    but "did it honour the schema" is the same question and deserves the same answer
    shape.
    """
    body = {
        "model": model_id,
        "max_tokens": 300,
        "messages": ASK,
        "tools": [
            {"name": "emit", "description": "Return.", "input_schema": SCHEMA}
        ],
        "tool_choice": {"type": "tool", "name": "emit"},
    }
    good = no_call = off = failed = 0
    for _ in range(samples):
        code, data = post(f"{prefix}/messages", body, headers=AV, **PROBE)
        if code != 200:
            failed += 1
            continue
        blocks = [b for b in (data.get("content") or [])
                  if isinstance(b, dict) and b.get("type") == "tool_use"]
        if not blocks:
            no_call += 1
            continue
        if len(blocks) > 1:
            # Same reasoning as _tool_conforms: a forced named tool means one call.
            off += 1
            continue
        block = blocks[0]
        obj = block.get("input")
        conforms = (
            isinstance(obj, dict)
            and set(SCHEMA["required"]) <= set(obj)
            and not (SCHEMA.get("additionalProperties") is False
                     and set(obj) - set(SCHEMA["properties"]))
        )
        good += conforms
        off += not conforms
    detail = []
    if no_call:
        detail.append(f"{no_call}x no tool_use")
    if off:
        detail.append(f"{off}x off-schema")
    if failed:
        detail.append(f"{failed}x non-200")
    return f"{good}/{samples}" + (f" ({' '.join(detail)})" if detail else "")


def _rate(path: str, body: dict, label: str, samples: int = 3) -> str:
    """Conformance as a rate over `samples` runs, not a single verdict.

    One sample is not enough. `openai.gpt-oss-120b` accepts a strict `json_schema`
    request and then honours it about one time in five -- the other runs come back
    as `**{ "summary": ...`, markdown-bolded and unparseable. A single probe that
    happened to land on the good run labelled it plainly "ok", which is the same
    class of error as reading a capability off one call.
    """
    good = off_schema = failed = 0
    for _ in range(samples):
        code, data = post(path, body, **PROBE)
        if code != 200:
            failed += 1
        elif _conforms(data) == "ok":
            good += 1
        else:
            off_schema += 1
    # Keep WHY the run did not conform. Collapsing everything to `good/samples`
    # made a 0/3 caused by three transient 5xx indistinguishable from a 0/3 caused
    # by three 200s carrying prose -- and the legend under the table asserts it is
    # the second. A rate whose meaning has to be taken on trust is the thing this
    # column was rebuilt to avoid.
    detail = []
    if off_schema:
        detail.append(f"{off_schema}x200 off-schema")
    if failed:
        detail.append(f"{failed}x non-200")
    suffix = f", {' '.join(detail)}" if detail else ""
    return f"{good}/{samples} ({label}{suffix})"


def _conforms(data: dict) -> str:
    """Did the RESULT honour the schema, not merely earn a 200?

    This distinction is the whole point of the column. `openai.gpt-oss-120b` returns
    200 for a strict `json_schema` request and then emits `**{ "summary": ...`, a
    markdown-bolded string that is not JSON. Measured over five runs it conformed
    1/5, and gpt-oss-20b 0/5, so a column that said "ok" on the strength of the
    status was reporting the opposite of what a caller needs to know.
    """
    # Read both response shapes without importing a helper: Responses nests text
    # blocks under output[].content[], Chat Completions puts it on the message.
    text = ((data.get("choices") or [{}])[0].get("message") or {}).get("content") or ""
    if not text:
        pieces = []
        for item in data.get("output") or []:
            for block in item.get("content") or []:
                if block.get("text"):
                    pieces.append(block["text"])
        text = "".join(pieces)
    text = (text or "").strip()
    try:
        obj = json.loads(text)
    except Exception:
        return "200 off-schema"
    if not isinstance(obj, dict) or not set(SCHEMA["required"]) <= set(obj):
        return "200 off-schema"
    # `required` alone is not the schema. With additionalProperties False, an object
    # carrying extra keys violates it, and checking only the required set scored that
    # as "honoured the schema" -- which is the exact confusion this column exists to
    # remove.
    if SCHEMA.get("additionalProperties") is False:
        if set(obj) - set(SCHEMA["properties"]):
            return "200 off-schema"
    return "ok"


def _native_schema(model_id: str, prefix: str) -> str:
    """Try Responses text.format, then fall back to Chat Completions.

    The two 400s here mean opposite things and must be told apart:

        "does not support the '/v1/responses' API"  -> wrong surface, try the other
        anything else                               -> Responses exists and refused
                                                       the schema

    An earlier version returned on ANY 400, so the eight Chat-Completions-only
    families never reached the fallback and the "Structured-output support" column
    reported them as having no native schema without ever asking the API they
    actually serve. The label now says which surface answered.
    """
    resp_body = {
        "model": model_id,
        "input": "Answer 'hi'.",
        "max_output_tokens": 300,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "s",
                "schema": SCHEMA,
                "strict": True,
            }
        },
    }
    code, data = post(f"{prefix}/responses", resp_body, **PROBE)
    if code == 200:
        return _rate(f"{prefix}/responses", resp_body, "Resp")
    message = err(data, limit=300).lower()
    lacks_responses = "does not support" in message and "api" in message
    if code == 400 and not lacks_responses:
        return "400 (Resp)"

    # This model has no Responses API: ask Chat Completions instead.
    cc_body = {
        "model": model_id,
        "messages": ASK,
        "max_tokens": 300,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "s", "strict": True, "schema": SCHEMA},
        },
    }
    code, data = post(f"{prefix}/chat/completions", cc_body, **PROBE)
    if code != 200:
        return f"{code} (CC)"
    return _rate(f"{prefix}/chat/completions", cc_body, "CC")


def _tool_conforms(data: dict) -> str:
    """Did the forced tool actually fire, with arguments that honour the schema?

    A 200 is not the answer here either, and for this route it is the worse trap:
    the notebook's own §3 records that `openai.gpt-oss-120b` returns 200 with a
    named `tool_choice` and makes ZERO calls. So "ok" on the strength of the status
    reported the mechanism as working precisely where it silently does nothing.
    """
    calls = ((data.get("choices") or [{}])[0].get("message") or {}).get("tool_calls")
    if not calls:
        return "200 no call"
    # A forced, NAMED tool_choice asks for one call. More than one means the model
    # ignored the constraint, which is the thing this column measures -- and inspecting
    # only calls[0] scored that as "ok".
    if len(calls) > 1:
        return f"200 {len(calls)} calls"
    raw = ((calls[0] or {}).get("function") or {}).get("arguments")
    try:
        # `arguments` is a JSON STRING per the OpenAI schema, but some models send the
        # object directly. Rejecting that as "bad args" mislabelled a conforming
        # answer.
        obj = raw if isinstance(raw, dict) else json.loads(raw or "")
    except Exception:
        return "200 bad args"
    if not isinstance(obj, dict) or not set(SCHEMA["required"]) <= set(obj):
        return "200 off-schema"
    if SCHEMA.get("additionalProperties") is False and (
            set(obj) - set(SCHEMA["properties"])):
        return "200 off-schema"
    return "ok"


def _forced_tool(model_id: str, prefix: str, samples: int = 3) -> str:
    """Forced function call — the portable structured-output route, as a rate.

    Reported the same way as the native-schema column beside it. It used to return a
    bare "ok" for HTTP 200 while its neighbour reported n/3 conformance, under one
    legend explaining the rate — so a reader following the legend read "ok" as
    3/3-equivalent for the route this notebook elsewhere calls the unreliable one.
    """
    body = {
        "model": model_id,
        "messages": ASK,
        "max_tokens": 300,
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "emit",
                    "description": "Return.",
                    "parameters": SCHEMA,
                },
            }
        ],
        "tool_choice": {"type": "function", "function": {"name": "emit"}},
    }
    good = no_call = off = failed = 0
    for _ in range(samples):
        code, data = post(f"{prefix}/chat/completions", body, **PROBE)
        if code != 200:
            failed += 1
            continue
        verdict = _tool_conforms(data)
        if verdict == "ok":
            good += 1
        elif verdict == "200 no call":
            no_call += 1
        else:
            off += 1
    detail = []
    if no_call:
        detail.append(f"{no_call}x no call")
    if off:
        detail.append(f"{off}x off-schema")
    if failed:
        detail.append(f"{failed}x non-200")
    return f"{good}/{samples}" + (f" ({' '.join(detail)})" if detail else "")

One dispatcher over the three probes above.

In [12]:
def probe_structured(model_id: str) -> dict:
    """Which structured-output mechanisms does this model actually accept?"""
    prefix = api_prefix(model_id)
    if prefix == "/anthropic/v1":
        return {
            "model": model_id,
            "native_schema": "n/a",
            "forced_tool": _claude_forced_tool(model_id, prefix),
        }
    return {
        "model": model_id,
        "native_schema": _native_schema(model_id, prefix),
        "forced_tool": _forced_tool(model_id, prefix),
    }


with cf.ThreadPoolExecutor(max_workers=4) as pool:
    struct_results = list(pool.map(probe_structured, REPRESENTATIVES))

print(f"{'model':40} {'native schema':>30} {'forced tool':>30}")
print("-" * 102)
for row in struct_results:
    print(f"{row['model']:40} {row['native_schema']:>30} {row['forced_tool']:>30}")
print()
print("  BOTH columns are n/3: how many of three runs produced a body that actually")
print("  honoured the schema. The forced-tool column used to be a bare status, so")
print("  'ok' sat under a legend explaining a rate. Anything below 3/3 needs")
print("  client-side validation, not trust.")
print()
print("  The parenthetical says WHY a run did not count, and the distinction")
print("  matters: '200 off-schema' is the model ignoring the constraint, 'no call'")
print("  is a forced tool that never fired, 'non-200' is not a capability answer")
print("  at all. A bare 0/3 cannot tell you which of the three you are looking at.")

# The caveat under this table used to name Gemma 4 and gpt-oss from memory, and
# contradicted these very rows. Derive it instead.
def _num(cell):
    """The conforming-run count, or None when the cell is not a rate at all."""
    text = str(cell)
    return int(text.split("/")[0]) if re.match(r"^\d+/\d+", text) else None


def _hard_failure(cell) -> bool:
    """A bare status like "400 (Resp)" -- the request was refused outright.

    `_num()` returns None for these, and the filter below excluded None so that
    "n/a" would not be listed. That silently excluded the hard refusals too, so a
    model whose schema request 400s printed "none on this run" under "models that did
    NOT honour the schema", directly below a table row reading `400 (Resp)`.
    """
    return bool(re.match(r"^\d{3}\b", str(cell))) and "/" not in str(cell)


imperfect_native = [(r["model"], r["native_schema"]) for r in struct_results
                    if (_num(r["native_schema"]) not in (None, 3)
                        or _hard_failure(r["native_schema"]))]
imperfect_tool = [(r["model"], r["forced_tool"]) for r in struct_results
                  if (_num(r["forced_tool"]) not in (None, 3)
                      or _hard_failure(r["forced_tool"]))]
print("\n  models that did NOT honour the schema on all three runs:")
for label, rows_ in (("native schema", imperfect_native), ("forced tool", imperfect_tool)):
    if rows_:
        for model, cell in rows_:
            print(f"    {label:14} {model:40} {cell}")
    else:
        print(f"    {label:14} none on this run")

model                                                     native schema                    forced tool
------------------------------------------------------------------------------------------------------
google.gemma-4-31b                                           3/3 (Resp)                            3/3
openai.gpt-5.6-sol                                           3/3 (Resp)               0/3 (3x non-200)
openai.gpt-oss-120b                          0/3 (CC, 3x200 off-schema)                            3/3
anthropic.claude-haiku-4-5                                          n/a                            3/3
xai.grok-4.3                               0/3 (Resp, 3x200 off-schema)                            3/3
qwen.qwen3-32b                                                 3/3 (CC)                            3/3
deepseek.v3.2                                                  3/3 (CC)               1/3 (2x no call)
zai.glm-5                                                      3/3 (CC)  

**Caveat worth remembering:** "accepted" is not "reliable". A request can return 200
while the model ignores the constraint, or appends characters after a valid JSON
object. Which models did that *on this run* is printed by the cell above, under
"models that did NOT honour the schema on all three runs" — read that list rather
than a list written here.

This paragraph used to name two: gpt-oss ignoring a named `tool_choice`, and Gemma 4
appending characters "roughly half the time". Both sat directly beneath rows that said
otherwise — `google.gemma-4-31b 3/3 (Resp)` and `openai.gpt-oss-120b … forced tool:
ok`. The rates move between runs, which is exactly why the sentence has to be derived
from them and not written alongside them. Always validate the payload you get back.

## 6. A reusable capabilities resolver

Everything above, distilled into something you can paste into a project.

In [13]:
CAPABILITIES_PY = '''
"""Resolve Bedrock request details per model and endpoint. From live probes.

Read the caveat before you rely on this. The constants below are a *snapshot*:
which models refuse which parameters has changed twice during this collection's
life. A resolver built on a static table is right until it is not, and it fails
closed in the worst way -- a 400 in production on a model you never re-tested.

The durable design is the one in `02-migrating-from-openai.ipynb` section 7: send
the request, and when the service returns a 400 that *names* a parameter, drop that
parameter and retry. Bedrock is consistent about naming it. Use this module for
routing (which path, which API, which budget field) and let error handling deal
with sampling.

Every routing function takes `endpoint="mantle"` or `endpoint="runtime"`, because
the path, the API surface and the model ID all differ between the two. It does NOT
translate model IDs -- ask the service, via `runtime_id_for()` in
`_shared/bedrock.py`, rather than encoding a mapping that will age.
"""
import re

OPENAI_PREFIX_FAMILIES = ("google.gemma-4", "xai.")

# The openai family splits on mantle, and this tuple used to read "openai.gpt-5" --
# a model GENERATION. GPT-6 Astra falsified that on 8 Sep 2026: it is not a gpt-5, so
# it fell through to bare /v1, where mantle refuses it by name, `model
# `openai.gpt-6-astra` isn't supported on this route`, while /openai/v1 answers 200.
#
# Keyed on the LINE instead of the version: the hosted GPT models serve /openai/v1 and
# the open-weight gpt-oss line serves bare /v1. Measured for the 11 openai models on
# mantle in us-east-1 and the 9 in us-west-2 on 9 Sep 2026, with 20/20 agreeing. Keyed this
# way a gpt-7 needs no edit here.
OPENAI_HOSTED_GPT = "openai.gpt-"
OPENAI_OPEN_WEIGHT = "openai.gpt-oss"

# A geo/global inference-profile prefix is not part of the family name, and
# bedrock-runtime requires one for several families. Strip it before matching.
PROFILE_PREFIX = re.compile(r"^(us|eu|apac|global|in)[.]")

# Families served by Chat Completions only -- the Responses API 400s for these.
# Note gpt-oss-safeguard is here while base gpt-oss is not: the provider prefix
# is not enough to decide.
CHAT_ONLY = ("qwen.", "deepseek.", "zai.", "minimax.", "moonshotai.", "mistral.",
             "nvidia.", "writer.", "openai.gpt-oss-safeguard", "google.gemma-3")

# Chat Completions wants max_completion_tokens rather than max_tokens for these, and
# 400s on max_tokens. A MEASURED LIST and not a rule: gpt-5.4 and gpt-5.5 take
# max_tokens, gpt-5.6 and gpt-6-astra reject it, and nothing in the ID predicts which
# side a new model lands on. Measured on both endpoints on 9 Sep 2026.
#
# The check below does NOT exercise this entry for either model, and that is worth
# knowing rather than assuming: surface() routes both to the Responses API, where the
# field is max_output_tokens, so the Chat Completions branch is never taken. It bites
# only when you ask for api="chat" explicitly. Probe before adding a model here.
COMPLETION_TOKENS_FAMILIES = ("openai.gpt-5.6", "openai.gpt-6-astra")

# Models that reject `temperature` and `top_p` outright ("deprecated"). Confirmed
# against the service, and the Claude Opus 4.7 model card states it: "Starting with
# Claude Opus 4.7, temperature, top_p, and top_k parameters are no longer supported."
#
# The Fable models are in here now, and the story of how is worth keeping, because the
# previous note in this spot was WRONG in a way that looked careful.
#
# It said fable-5 "could not be probed: every call returns `400 data retention mode
# 'default' is not available for this model`, with or without sampling parameters, so
# the account cannot reach it to find out" -- and declined to guess. Declining to guess
# was right. Concluding the model was unreachable was not: that 400 is
# **bedrock-mantle only**. On bedrock-runtime the same model answers. Measured
# 7 Sep 2026 on /anthropic/v1/messages:
#
#                                     mantle                    runtime
#   anthropic.claude-fable-5          400 data retention        200
#   anthropic.claude-fable-5-1        404 does not exist        200
#
# Fable is a Covered Model in Anthropic's terms, which is why mantle refuses it under
# the default retention mode; AWS documents the extra retention, safety-review and
# access policy that comes with that designation, and Enterprise Frontier Safeguards
# as the way to use it. So "cannot be probed" was really "cannot be probed on the
# endpoint I happened to try".
#
# Probed on runtime, both Fables behave exactly like Opus 5:
#
#   temperature=0.5   400 `temperature` is deprecated for this model.
#   temperature=1.0   200
#   top_p=0.9         400 `top_p` is deprecated for this model.
#
# Hence both belong here. The lesson is the one this collection keeps relearning: a
# negative result carries the conditions it was measured under, and "unreachable" needs
# the endpoint named or it is not a finding.
NO_SAMPLING = ("anthropic.claude-opus-5", "anthropic.claude-sonnet-5",
               "anthropic.claude-opus-4-8", "anthropic.claude-opus-4-7",
               "anthropic.claude-fable-5")
# Models that accept EITHER temperature or top_p but not both in one request.
ONE_SAMPLING_PARAM = ("anthropic.claude-haiku-4-5",)
# Models that accept `temperature` ONLY at its default 1.0, and reject `top_p`.
# The restriction is per (model, API), not per model. Measured on 22 Aug 2026, and
# re-measured for the whole hosted line on 9 Sep 2026:
#
#                    Responses          Chat Completions
#   gpt-6-astra      temperature=1.0    temperature=1.0     only the default
#   gpt-5.6          temperature=1.0    temperature=1.0     only the default
#   gpt-5.5          temperature=1.0    any value           API-dependent
#   gpt-5.4          any value          any value           unrestricted
#
# Astra is newer than gpt-5.5 and gpt-5.4 yet more restricted than both, so "newer
# is looser" is not a rule either. Each row here is a measurement.
#
# So the rule needs BOTH keys. Keying on the model alone was wrong in each
# direction in turn: with gpt-5.5 included, sampling() discarded a temperature the
# Chat Completions route would have honoured; with it removed, the Responses route
# 400ed. gpt-5.4 is deliberately absent from both tuples.
TEMPERATURE_DEFAULT_ONLY = ("openai.gpt-5.6",
                            "openai.gpt-6-astra")         # restricted on every API
TEMPERATURE_DEFAULT_ON_RESPONSES = ("openai.gpt-5.5",)    # restricted on Responses
# Models that accept flex/priority service tiers. Not a Messages parameter at all.
TIERED = ("openai.gpt-oss", "google.gemma-", "xai.", "qwen.", "deepseek.",
          "zai.", "minimax.", "moonshotai.", "mistral.", "nvidia.", "writer.")

# On bedrock-runtime the Responses API reaches only these families; everything
# else OpenAI-compatible there is Chat Completions. Much narrower than on mantle.
RUNTIME_RESPONSES = ("openai.gpt-5.6", "openai.gpt-6-astra", "xai.grok-4.6")

# Features that exist on exactly one endpoint. Check before you pick.
MANTLE_ONLY_FEATURES = ("server_side_tools", "web_search", "background",
                        "projects", "workspaces")
RUNTIME_ONLY_FEATURES = ("guardrails", "prompt_routing", "cross_region",
                         "provisioned_throughput", "batch")


def api_prefix(model_id, endpoint="mantle"):
    """Return the URL prefix that serves this model's inference APIs.

    `endpoint` is "mantle" or "runtime", and it changes the answer: runtime serves
    every OpenAI-compatible model on /openai/v1 and has no /v1 inference path.

    Validated rather than defaulted. `api_prefix(m, "runtiem")` used to return "/v1"
    with no complaint, which is the worst possible answer: a path that exists on one
    endpoint and not the other.
    """
    if endpoint not in ("mantle", "runtime"):
        raise ValueError(f"endpoint must be 'mantle' or 'runtime', got {endpoint!r}")
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith("anthropic."):
        return "/anthropic/v1"
    if endpoint == "runtime":
        return "/openai/v1"
    if bare.startswith(OPENAI_HOSTED_GPT):
        # gpt-oss is the open-weight line and sits on bare /v1; every other
        # openai.gpt-* is hosted and sits on /openai/v1.
        return "/v1" if bare.startswith(OPENAI_OPEN_WEIGHT) else "/openai/v1"
    if bare.startswith(OPENAI_PREFIX_FAMILIES):
        return "/openai/v1"
    return "/v1"


# Claude models MEASURED to serve /anthropic/v1/messages on bedrock-runtime, as of
# 7 Sep 2026, addressed with a geo prefix. See surface() for why this is a list and
# not a rule.
RUNTIME_MESSAGES_CLAUDE = (
    "anthropic.claude-fable-5",          # covers fable-5 and fable-5-1
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-7",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
)


def surface(model_id, endpoint="mantle"):
    """Which API this model actually serves: messages / chat / responses / converse.

    Narrower on bedrock-runtime, where Responses reaches the GPT-5.6, GPT-6 Astra and
    Grok 4.6 profiles and no others measured here, and SOME Claude models serve
    Messages.

    That last point has now been wrong twice, in opposite directions, and the second
    time is the instructive one.

    Version one returned "messages" for every `anthropic.` model on both endpoints,
    and 404ed on runtime. Version two inferred a rule from the single counter-example
    to hand: haiku-4-5 was the only Claude whose runtime ID was a legacy dated
    `-YYYYMMDD-vN:0` form, it 404ed, so "dated means Converse" looked like the split.
    One example, one rule.

    Re-measured 7 Sep 2026 over all 41 Claude IDs on runtime, that rule fails nine
    times, in both directions:

        us.anthropic.claude-haiku-4-5-20251001-v1:0   dated,     Messages 200
        us.anthropic.claude-sonnet-4-6                not dated, Messages 404
        us.anthropic.claude-opus-4-6-v1               not dated, Messages 404

    haiku-4-5's dated ID now works -- the service changed under a finding that was
    correct when written. And two models with short profile IDs do NOT serve Messages,
    so ID shape never was the mechanism. It is not chronological either: sonnet-4-6
    (Feb 2026) 404s while haiku-4-5 (Oct 2025) answers.

    What DOES hold, measured over the 15 `us.*` Claude IDs in `00-foundations/04` §8c:
    **every model that serves Messages also serves Converse, and four serve Converse
    only.** Not "Converse serves everything" -- four very old IDs (claude-3-haiku,
    claude-3-sonnet, opus-4-1, sonnet-4) serve neither, so a Converse route is not a
    guarantee either. The useful relation is the containment: Messages-capable is a
    strict subset of Converse-capable, so choosing Converse never loses you a model,
    and choosing Messages can.

    Hence Converse is the default on runtime, and Messages is an opt-in for models
    measured to support it -- because Messages carries things Converse does not expose
    as first-class, notably the beta headers for computer use and context management.

    The subset is a LIST, not a rule, and a list goes stale. `00-foundations/04`
    probes it live, and `harness/quality/scripts/verify-live-claims.py` fails if this
    tuple and the service disagree. A bare ID with no geo prefix returns 400 on
    runtime Messages whatever the model, so it is routed to Converse too.
    """
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith("anthropic."):
        if endpoint != "runtime":
            return "messages"
        # Runtime: Converse unless this model is measured to serve Messages, AND the
        # caller addressed it with a geo profile. A bare ID 400s on Messages.
        has_geo = PROFILE_PREFIX.match(model_id) is not None
        if has_geo and bare.startswith(RUNTIME_MESSAGES_CLAUDE):
            return "messages"
        return "converse"
    if endpoint == "runtime":
        return "responses" if bare.startswith(RUNTIME_RESPONSES) else "chat"
    if bare.startswith(CHAT_ONLY):
        return "chat"
    return "responses"


def inference_path(model_id, api="auto", endpoint="mantle"):
    """Full inference path, e.g. "/v1/chat/completions".

    `api="auto"` picks the surface the model actually serves. Defaulting to
    /responses for everything non-Anthropic -- which an earlier version of this
    module did -- 400s for every Chat-Completions-only family.
    """
    if api not in ("auto", "messages", "chat", "responses", "converse"):
        raise ValueError(
            f"api must be auto/messages/chat/responses/converse, got {api!r}. "
            "Note the second positional argument here is `api`, not `endpoint`: "
            "inference_path(m, 'runtime') is a bug that used to return /v1/responses "
            "silently. Pass endpoint= by keyword."
        )
    prefix = api_prefix(model_id, endpoint)
    chosen = surface(model_id, endpoint) if api == "auto" else api
    if chosen == "converse":
        # Not an OpenAI- or Anthropic-shaped path: Converse is its own operation on
        # bedrock-runtime, and the token budget moves into inferenceConfig.
        return f"/model/{model_id}/converse"
    if chosen == "messages":
        return prefix + "/messages"
    if chosen == "chat":
        return prefix + "/chat/completions"
    return prefix + "/responses"


def token_limit_field(model_id, api="auto", endpoint="mantle"):
    """The budget parameter this model+API expects.

    Getting this wrong yields a 400 that reads exactly like "this API does not
    exist here", which is a trap worth avoiding by construction.
    """
    if api not in ("auto", "messages", "chat", "responses", "converse"):
        raise ValueError(
            f"api must be auto/messages/chat/responses/converse, got {api!r}. "
            "Note the second positional argument here is `api`, not `endpoint`: "
            "token_limit_field(m, 'runtime') silently answered as though you had "
            "named an API. Pass endpoint= by keyword."
        )
    chosen = surface(model_id, endpoint) if api == "auto" else api
    if chosen == "converse":
        return "inferenceConfig.maxTokens"  # nested, not a top-level field
    if chosen == "responses":
        return "max_output_tokens"          # minimum 16
    if chosen == "chat" and PROFILE_PREFIX.sub("", model_id).startswith(
            COMPLETION_TOKENS_FAMILIES):
        return "max_completion_tokens"
    return "max_tokens"                     # Messages: required, no default


def sampling(model_id, temperature=None, top_p=None, api="auto", endpoint="mantle"):
    """Drop or clamp parameters this model rejects, instead of earning a 400.

    Match on the model ID with any geo/global prefix removed. Matching the raw ID
    is a real bug and a quiet one: `us.anthropic.claude-opus-5` does not start with
    `anthropic.claude-opus-5`, so every rule below stops applying the moment you
    move to bedrock-runtime, and the 400 you get back names `temperature` on a
    model this table already knew rejects it. The verification cell below is what
    found that.

    One of these restrictions is per (model, API) rather than per model. Measured on
    bedrock-mantle, `temperature=0.5` and `top_p=0.9`:

        openai.gpt-5.4      Responses 200   Chat Completions 200
        openai.gpt-5.5      Responses 400   Chat Completions 200
        openai.gpt-5.6-sol  Responses 400   Chat Completions 400

    So `api` is the axis that matters, and `endpoint` is only a way of guessing it.
    Passing `endpoint` alone could not express the gpt-5.5 row: `surface()` maps
    gpt-5.5 to "responses" on both endpoints, so `sampling("openai.gpt-5.5",
    temperature=0.7, top_p=0.95)` silently returned {} and dropped both parameters
    on the one route that honours them. Name the API when you know it:

        sampling("openai.gpt-5.5", temperature=0.7, api="chat")      -> both kept
        sampling("openai.gpt-5.5", temperature=0.7, api="responses") -> both dropped

    `api="auto"` keeps the old behaviour of asking surface(), which is right whenever
    you are letting inference_path() choose the route too.
    """
    if api not in ("auto", "messages", "chat", "responses", "converse"):
        raise ValueError(
            f"api must be auto/messages/chat/responses/converse, got {api!r}."
        )
    bare = PROFILE_PREFIX.sub("", model_id)
    if bare.startswith(NO_SAMPLING):
        return {}
    chosen = surface(model_id, endpoint) if api == "auto" else api
    on_responses = chosen == "responses"
    pinned = bare.startswith(TEMPERATURE_DEFAULT_ONLY) or (
        on_responses and bare.startswith(TEMPERATURE_DEFAULT_ON_RESPONSES)
    )
    out = {}
    if temperature is not None:
        if pinned:
            if abs(float(temperature) - 1.0) < 1e-9:
                out["temperature"] = 1.0
            # else omit entirely rather than 400
        else:
            out["temperature"] = temperature
    if top_p is not None and not pinned:
        out["top_p"] = top_p
    # "`temperature` and `top_p` cannot both be specified for this model."
    if bare.startswith(ONE_SAMPLING_PARAM) and len(out) == 2:
        out.pop("top_p")
    return out


def service_tier(model_id, tier="default", endpoint="mantle"):
    """Downgrade to 'default' where flex/priority are unsupported.

    Returns None for Messages -- omit the field -- but not for the reason this
    docstring used to give. It said "service_tier is not a parameter at all" there,
    which is measured false: Messages ACCEPTS the parameter and refuses every value
    except "default". Probed on anthropic.claude-haiku-4-5 via
    /anthropic/v1/messages:

        (omitted)              200   usage.service_tier='standard'
        service_tier=default   200   usage.service_tier='standard'
        service_tier=standard  400   unsupported service_tier 'standard'
        service_tier=flex      400   unsupported service_tier 'flex'
        service_tier=priority  400   unsupported service_tier 'priority'

    Note the last twist: the response reports "standard", a value the request cannot
    set. Omitting is still the right move, because "default" is the only accepted
    value and it is also the default -- but a caller who reads "not a parameter" will
    not understand the 400 they get when they try one of the others.

    Two bugs lived here, both of the kind sampling()'s docstring warns about one
    function above. It matched on the RAW id, so every geo-prefixed id -- which is
    to say every bedrock-runtime id -- fell through to "default" and silently lost
    the tier the caller asked for: service_tier("xai.grok-4.6", "flex") gave "flex"
    and service_tier("us.xai.grok-4.6", "flex") gave "default". And it called
    surface() without the endpoint, so a model that routes to Converse on runtime
    was judged by its mantle surface.
    """
    bare = PROFILE_PREFIX.sub("", model_id)
    if surface(model_id, endpoint) in ("messages", "converse"):
        return None
    if tier == "default" or bare.startswith(TIERED):
        return tier
    return "default"
'''

with open("capabilities.py", "w") as handle:
    handle.write(CAPABILITIES_PY)

sys.path.insert(0, ".")
# reload, not a bare import: `import` is a no-op when the module is already in
# sys.modules, so re-running this cell after editing CAPABILITIES_PY would write the
# new file and then print the OLD resolver's answers.
import importlib

import capabilities

capabilities = importlib.reload(capabilities)

print(f"{'model':32} {'path':30} {'budget field':22} sampling")
print("-" * 118)
for mid in (
    "google.gemma-4-31b",
    "xai.grok-4.3",
    "openai.gpt-5.6-sol",
    "openai.gpt-6-astra",
    "openai.gpt-oss-120b",
    "openai.gpt-oss-safeguard-20b",
    "anthropic.claude-sonnet-5",
    "anthropic.claude-haiku-4-5",
    "qwen.qwen3-32b",
):
    print(
        f"{mid:32} {capabilities.inference_path(mid):30} "
        f"{capabilities.token_limit_field(mid):22} "
        f"{json.dumps(capabilities.sampling(mid, temperature=0.7, top_p=0.95))}"
    )

model                            path                           budget field           sampling
----------------------------------------------------------------------------------------------------------------------
google.gemma-4-31b               /openai/v1/responses           max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
xai.grok-4.3                     /openai/v1/responses           max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
openai.gpt-5.6-sol               /openai/v1/responses           max_output_tokens      {}
openai.gpt-6-astra               /openai/v1/responses           max_output_tokens      {}
openai.gpt-oss-120b              /v1/responses                  max_output_tokens      {"temperature": 0.7, "top_p": 0.95}
openai.gpt-oss-safeguard-20b     /v1/chat/completions           max_tokens             {"temperature": 0.7, "top_p": 0.95}
anthropic.claude-sonnet-5        /anthropic/v1/messages         max_tokens             {}
anthropic.claude-haiku-

In [14]:
# Verify the resolver against the live endpoints. A resolver that is wrong is
# worse than none: it turns "read the docs" into "debug a 400 in production".
# So exercise it on BOTH endpoints and count the failures.
namespace: dict = {}
# CAPABILITIES_PY is the literal defined in the cell above, not input; this cell
# exists to prove that literal actually runs.
# nosemgrep: exec-detected
exec(CAPABILITIES_PY, namespace)  # nosec B102  # noqa: S102 - our own literal above

resolved_path = namespace["inference_path"]
resolved_field = namespace["token_limit_field"]
resolved_sampling = namespace["sampling"]
# service_tier() had ZERO callers anywhere in the repository while this cell's
# comment insisted that every special-cased family must appear here or its rule is
# never exercised. So it is called below, and its answer is sent.
resolved_tier = namespace["service_tier"]
resolved_surface = namespace["surface"]

# Every family whose behaviour the resolver special-cases must appear here, or the
# rule for it is never exercised. gpt-5.5 and gpt-5.4 were missing, and the
# TEMPERATURE_DEFAULT_ONLY rule was wrong about gpt-5.5 for exactly as long: the
# table below could not have caught it, because it never called the model.
#
# gpt-6-astra is here for the same reason. Four of the tuples above were keyed on
# "openai.gpt-5" or "openai.gpt-5.6" and so said nothing about it; with the old keys
# this row sent max_tokens and temperature=0.7 to a model that rejects both, and
# earned two 400s. Astra is not on mantle in us-east-1, so the mantle half of its
# row reports "not in region" rather than pretending to a measurement.
CHECK = [
    "openai.gpt-5.6-sol",
    "openai.gpt-6-astra",
    "openai.gpt-5.5",
    "openai.gpt-5.4",
    "openai.gpt-oss-120b",
    "anthropic.claude-opus-5",
    "anthropic.claude-haiku-4-5",
    "qwen.qwen3-32b",
    "deepseek.v3.2",
    "zai.glm-5",
    "google.gemma-4-31b",
    "xai.grok-4.6",
]
AV = {"anthropic-version": "2023-06-01"}
print(f"{'model':26} {'endpoint':8} {'resolved path':30} {'field':22} result")
print("-" * 104)

failures = []
tier_calls: list[tuple] = []
for model_id in CHECK:
    for endpoint in ("mantle", "runtime"):
        if endpoint == "mantle":
            send_id = model_id if model_id in models else None
        else:
            send_id = runtime_id_for(model_id, REGION)
        if send_id is None:
            print(f"{model_id:26} {endpoint:8} {'-- not on this endpoint --':30} "
                  f"{'-':22} skipped")
            continue

        path = resolved_path(send_id, endpoint=endpoint)
        field = resolved_field(send_id, endpoint=endpoint)
        # Converse is a boto3 operation rather than an HTTP path this cell can POST,
        # so a Converse-routed model is reported rather than called. It is not a
        # resolver failure: routing it away from /anthropic/v1/messages is the fix.
        if path.endswith("/converse"):
            print(f"{model_id:26} {endpoint:8} {'Converse (see ../00-foundations/04)':30} "
                  f"{field:22} routed")
            continue
        api = resolved_surface(send_id, endpoint)
        # Ask for "flex" every time and send whatever the resolver hands back. A
        # model the TIERED tuple excludes gets "default"; Messages gets None, which
        # must be OMITTED rather than serialised as JSON null -- a caller writing
        # body["service_tier"] = service_tier(...) would send null and earn a 400,
        # so the omission is part of the contract and is now demonstrated.
        tier = resolved_tier(send_id, "flex", endpoint=endpoint)
        body = {
            "model": send_id,
            field: 2000,
            # `api=api`: the temperature restriction is per (model, API), and
            # letting sampling() re-derive the surface cannot express gpt-5.5 on
            # Chat Completions.
            **resolved_sampling(send_id, 0.5, api=api, endpoint=endpoint),
        }
        if tier is not None:
            body["service_tier"] = tier
        if path.endswith("/responses"):
            body["input"] = "Reply OK"
        else:
            body["messages"] = [{"role": "user", "content": "Reply OK"}]

        caller = post if endpoint == "mantle" else runtime_post
        code, data = caller(
            path, body, region=REGION,
            headers=AV if path.endswith("/messages") else None,
            attempts=1, timeout=120,
        )
        good = ok(code, data)
        if not good:
            failures.append((model_id, endpoint, code, err(data)[:60]))
        print(f"{model_id:26} {endpoint:8} {path:30} {field:22} "
              f"tier={str(tier):7} "
              f"{'ok' if good else str(code) + ' ' + err(data)[:34]}")
        tier_calls.append((model_id, endpoint, api, tier))

print()
if failures:
    print(f"=> {len(failures)} combination(s) failed. The resolver is wrong for these,")
    print("   which is exactly the situation this cell exists to catch:")
    for model_id, endpoint, code, message in failures:
        print(f"     {model_id} on {endpoint}: {code} {message}")
else:
    # Count what actually happened. "every resolved request succeeded, on both
    # endpoints, for the 11 models above" was false as printed: five combinations were
    # skipped because the model is not on that endpoint, and one was routed to Converse
    # without being called, so four of the eleven were tested on one endpoint only.
    both = sorted({m for m, _e, _a, _t in tier_calls
                   if len({e for mm, e, _a2, _t2 in tier_calls if mm == m}) == 2})
    one = sorted({m for m, _e, _a, _t in tier_calls} - set(both))
    print(f"=> {len(tier_calls)} resolved request(s) sent, all succeeded. "
          f"{len(both)} model(s) were exercised on BOTH endpoints "
          f"({len(one)} on one only, and "
          f"{len(CHECK) - len(both) - len(one)} not reached at all).")
    print("   That is the bar for pasting this into a project. Re-run it when you")
    print("   add a model, and treat a failure here as a resolver bug rather than")
    print("   a service problem.")

# The check above is ONE-DIRECTIONAL, and that is a real gap. It sends what the
# resolver KEEPS and confirms the service accepts it -- so it catches a model wrongly
# ABSENT from a restriction tuple. It cannot catch one wrongly PRESENT: if a tuple
# claims a model refuses `temperature` when it does not, sampling() drops the
# parameter, the request succeeds, and this cell prints ok. That is precisely the
# gpt-5.5 bug the resolver's own docstring narrates.
#
# So also test the drops. But the refutation depends on WHY the parameter was dropped,
# and getting that wrong published a false claim: an earlier version of this cell sent
# every dropped parameter on its own and expected a 4xx, which printed
#
#     anthropic.claude-haiku-4-5  messages  top_p  200 -- RULE IS STALE
#
# for a rule that is correct. haiku-4-5 is in ONE_SAMPLING_PARAM, which says it accepts
# either temperature or top_p but NOT BOTH; `top_p` alone is supposed to return 200.
# The check was refuting a claim the resolver never made.
#
# So each restriction gets the test that would actually falsify it:
#
#   NO_SAMPLING            send the parameter alone      -> must be refused
#   ONE_SAMPLING_PARAM     send BOTH together            -> must be refused
#   TEMPERATURE_DEFAULT_*  send temperature != 1.0       -> must be refused
print("\nreverse check -- each restriction gets the test that would falsify IT:")
one_param = set(namespace["ONE_SAMPLING_PARAM"])
no_sampling = set(namespace["NO_SAMPLING"])
stale = []


def _send(send_id, endpoint, api, extra):
    path_ = resolved_path(send_id, api=api, endpoint=endpoint)
    field_ = resolved_field(send_id, api=api, endpoint=endpoint)
    body = {"model": send_id, field_: 2000, **extra}
    if path_.endswith("/responses"):
        body["input"] = "Reply OK"
    else:
        body["messages"] = [{"role": "user", "content": "Reply OK"}]
    caller = post if endpoint == "mantle" else runtime_post
    return caller(path_, body, region=REGION,
                  headers=AV if path_.endswith("/messages") else None,
                  attempts=2, timeout=120)


for model_id, endpoint, api, _tier in tier_calls:
    send_id = model_id if endpoint == "mantle" else runtime_id_for(model_id, REGION)
    if send_id is None:
        continue
    bare = re.sub(r"^(us|eu|apac|in|global)\.", "", send_id)
    kept = resolved_sampling(send_id, 0.5, top_p=0.9, api=api, endpoint=endpoint)
    dropped = [k for k in ("temperature", "top_p") if k not in kept]
    if not dropped:
        continue

    if any(bare.startswith(p) for p in one_param):
        rule, extra, why = ("ONE_SAMPLING_PARAM", {"temperature": 0.5, "top_p": 0.9},
                            "both together")
    elif any(bare.startswith(p) for p in no_sampling):
        rule, extra, why = ("NO_SAMPLING", {dropped[0]: 0.5}, f"{dropped[0]} alone")
    else:
        rule, extra, why = ("TEMPERATURE_DEFAULT", {"temperature": 0.5},
                            "temperature=0.5")

    code, data = _send(send_id, endpoint, api, extra)
    if code == 200:
        stale.append((model_id, endpoint, api, rule, why))
        verdict = f"200 -- {rule} IS STALE, the model accepts {why}"
    elif 400 <= code < 500:
        verdict = f"{code} refused -- {rule} confirmed"
    else:
        verdict = f"{code} inconclusive, re-run"
    print(f"  {model_id:26} {endpoint:8} {api:10} {why:16} {verdict}")

if stale:
    print(f"\n!! {len(stale)} restriction(s) the service does not agree with:")
    for row in stale:
        print(f"     {row}")
    print("   Re-probe and fix the tuple; the resolver is discarding control the")
    print("   model would have honoured.")
else:
    print("  => every restriction was falsifiable and survived. Both directions hold.")

# And the Messages short-circuit itself. service_tier() returns None for Messages, and
# the reason matters: five places in this collection used to say "not a Messages
# parameter at all", which is false. Measure it.
print("\nservice_tier on /anthropic/v1/messages -- which values does it accept?")
CLAUDE_TIER_PROBE = next((m for m in CHECK
                          if m.startswith("anthropic.") and m in models), None)
if CLAUDE_TIER_PROBE is None:
    print("  no Claude model available on mantle in this Region; skipped")
else:
    accepted, refused = [], []
    for value in (None, "default", "standard", "flex", "priority"):
        body = {"model": CLAUDE_TIER_PROBE, "max_tokens": 16,
                "messages": [{"role": "user", "content": "Hi"}]}
        if value is not None:
            body["service_tier"] = value
        code, data = post("/anthropic/v1/messages", body, region=REGION,
                          headers=AV, attempts=2, timeout=60)
        label = "(omitted)" if value is None else value
        if code == 200:
            accepted.append(label)
            reported = (data.get("usage") or {}).get("service_tier")
            print(f"  {label:12} 200  usage.service_tier={reported!r}")
        else:
            refused.append(label)
            print(f"  {label:12} {code}  {err(data)[:60]}")
    print(f"\n  => accepted: {accepted}   refused: {refused}")
    print("     The parameter is RECOGNISED -- the 400 names it -- and only `default`")
    print("     is accepted, which is also the default. So omit it. That is why the")
    print("     resolver returns None here, and it is not the same as the parameter")
    print("     being unknown.")
    if accepted and refused:
        print(f"     Note the twist: the response reports service_tier="
              f"'standard' while sending 'standard' is refused.")

# What service_tier() decided, so the TIERED tuple and the Messages short-circuit are
# visible rather than merely present in the source.
# `tier_calls` is appended for every route that was CALLED, and every one of those
# succeeded or it would be in `failures` above -- but say so from `failures`, not from
# a sentence that would keep claiming "accepted" if it did not.
_tier_ok = "all accepted" if not failures else f"{len(failures)} FAILED, see above"
print(f"\nservice_tier(model, 'flex') per route -- {len(tier_calls)} sent, "
      f"{_tier_ok}:")
for model_id, endpoint, api, tier in tier_calls:
    reason = ("omitted: Messages refuses every value but default" if tier is None
              else "downgraded: family not in TIERED" if tier == "default"
              else "kept")
    print(f"  {model_id:26} {endpoint:8} {api:10} -> {str(tier):8} {reason}")
downgraded = [m for m, _, _, t in tier_calls if t == "default"]
omitted = [m for m, _, _, t in tier_calls if t is None]
print(f"  kept 'flex' for {len([1 for *_, t in tier_calls if t == 'flex'])} route(s), "
      f"downgraded {len(downgraded)}, omitted {len(omitted)}")

model                      endpoint resolved path                  field                  result
--------------------------------------------------------------------------------------------------------


openai.gpt-5.6-sol         mantle   /openai/v1/responses           max_output_tokens      tier=default ok


openai.gpt-5.6-sol         runtime  /openai/v1/responses           max_output_tokens      tier=default ok
openai.gpt-6-astra         mantle   -- not on this endpoint --     -                      skipped


openai.gpt-6-astra         runtime  /openai/v1/responses           max_output_tokens      tier=default ok


openai.gpt-5.5             mantle   /openai/v1/responses           max_output_tokens      tier=default ok
openai.gpt-5.5             runtime  -- not on this endpoint --     -                      skipped


...

  openai.gpt-6-astra         runtime  responses  temperature=0.5  400 refused -- TEMPERATURE_DEFAULT confirmed
  openai.gpt-5.5             mantle   responses  temperature=0.5  400 refused -- TEMPERATURE_DEFAULT confirmed
  anthropic.claude-opus-5    mantle   messages   temperature alone 400 refused -- NO_SAMPLING confirmed


  anthropic.claude-opus-5    runtime  messages   temperature alone 400 refused -- NO_SAMPLING confirmed
  anthropic.claude-haiku-4-5 mantle   messages   both together    400 refused -- ONE_SAMPLING_PARAM confirmed


  anthropic.claude-haiku-4-5 runtime  messages   both together    400 refused -- ONE_SAMPLING_PARAM confirmed
  => every restriction was falsifiable and survived. Both directions hold.

service_tier on /anthropic/v1/messages -- which values does it accept?


  (omitted)    200  usage.service_tier='standard'


  default      200  usage.service_tier='standard'
  standard     400  unsupported service_tier 'standard'
  flex         400  unsupported service_tier 'flex'
  priority     400  unsupported service_tier 'priority'

  => accepted: ['(omitted)', 'default']   refused: ['standard', 'flex', 'priority']
     The parameter is RECOGNISED -- the 400 names it -- and only `default`
     is accepted, which is also the default. So omit it. That is why the
     resolver returns None here, and it is not the same as the parameter
     being unknown.
     Note the twist: the response reports service_tier='standard' while sending 'standard' is refused.

service_tier(model, 'flex') per route -- 19 sent, all accepted:
  openai.gpt-5.6-sol         mantle   responses  -> default  downgraded: family not in TIERED
  openai.gpt-5.6-sol         runtime  responses  -> default  downgraded: family not in TIERED
  openai.gpt-6-astra         runtime  responses  -> default  downgraded: family not in TIERED
  openai.g

## 7. A decision guide

| If you need… | Use |
|---|---|
| Web Search grounding | the hosted `openai.gpt-*` models (Responses) — the family measured to serve it here. Confirmed for `gpt-6-astra` on 9 Sep 2026 |
| Reasoning traces you can read | Responses API: gemma-4 (at `effort="high"`), gpt-5.x, gpt-oss. Grok's is **encrypted**. On Chat Completions several `/v1` families put it in the non-standard `message.reasoning` |
| Server-side tools (Lambda / Gateway) | Responses API models; built-ins on gpt-oss |
| Adaptive thinking + 1h prompt cache | `anthropic.claude-*` (Messages) |
| Explicit prompt-cache breakpoints | `openai.gpt-5.6-*` |
| Server-side conversation state | Responses API + `store=True` |
| Zero data retention | any model, `store=False` + retention mode `none` |
| EU data residency | check `eu-central-1` — no Anthropic, no gpt-5.x, no xAI |
| Vision | mantle: gemma-4, gemma-3, qwen3-vl, nemotron-nano-12b, palmyra-vision, Claude. runtime via Converse: nova-lite/pro/2-lite, llama4-scout/maverick, gemma-3, Claude. Each is demonstrated with a known-answer image in its family notebook |
| Cheapest viable model | Ministral / Nemotron Nano / GLM Flash ladders |
| Fine-tuning | `gpt-oss-20b` or `qwen3-32b`, us-west-2 only |

## Gotchas this survey exposes

| Gotcha | Detail |
|---|---|
| Three path prefixes | And a split *within* the OpenAI family |
| Responses API coverage | Only a minority of families — and the gpt-oss **safeguard** variants lack it while base gpt-oss has it |
| Budget field name | Chat Completions wants `max_completion_tokens` on gpt-5.6. That 400 reads like a missing API |
| Claude is Messages-only | Both OpenAI-compatible APIs 400. And on `bedrock-runtime`, Messages serves only **some** Claude models — a subset no ID shape predicts. Measured across all 41 runtime Claude IDs: `us.anthropic.claude-sonnet-4-6` and `us.anthropic.claude-opus-4-6-v1` are short profile IDs that 404, while the dated `us.anthropic.claude-haiku-4-5-20251001-v1:0` answers 200. A bare ID with no geo prefix 400s whatever the model. Every model that serves Messages also serves **Converse**, and four serve Converse only — so Converse never loses you a model that Messages would have reached. Reach for it on runtime unless you need a Messages-only feature, and probe the model you intend to call. §8c of `../00-foundations/04` measures the split |
| No universal sampling config | On Responses, gpt-5.5 and gpt-5.6 want `temperature=1.0` and refuse `top_p`; on Chat Completions only gpt-5.6 is restricted, and gpt-5.4 is unrestricted on both; newer Claude refuses both; `haiku-4-5` takes either but not both |
| Tier support varies | gpt-5.x is `default`-only |
| Wrong path may stall | Always set a client-side timeout when probing |
| 200 ≠ honoured | Constraints can be silently ignored — validate output |
| Region footprint | 55 models in us-east-1, 33 in eu-central-1 |

## Next
- `02-migrating-from-openai.ipynb` — porting an existing OpenAI codebase
- `03-production-hardening.ipynb` — the pre-launch checklist